# 01 — Echo smoke

End-to-end check that the egras stack works for a realistic B2B integration,
demonstrated via the typed `egras-client` library:

1. Operator logs in (`Client.login`).
2. Operator provisions a new tenant org (`tenants.create_organisation`).
3. Operator switches into the tenant org (`security.switch_org`) and creates a service account.
4. Operator grants the service account `org_admin` (which carries `echo:invoke` after migration 0014).
5. Operator mints an API key restricted to `scopes=['echo:invoke']`.
6. The service-account caller hits `POST /api/v1/echo` and asserts the payload round-trips with the right org and key id.

Two negative checks follow:

7. A second key without `echo:invoke` is rejected with 403 (caught as `egras_client.errors.Forbidden`).
8. The operator overrides the org's `auth.api_key_headers` flag to `['x-api-key']` only; the same key sent via `Authorization: Bearer` returns 401, while `X-API-Key` still works.

In [ ]:
import os, time, httpx

from egras_client import Client
from egras_client.errors import Forbidden, Unauthorized
from egras_client.helpers import operator_credentials
from egras_client.models import (
    AddUserToOrganisationRequest,
    CreateApiKeyRequest,
    CreateOrganisationRequest,
    CreateServiceAccountRequest,
    SwitchOrgRequest,
)

BASE = os.environ.get('EGRAS_BASE_URL', 'http://localhost:8080')

## Operator login + org provisioning

In [ ]:
email, password = operator_credentials()
email, password

In [ ]:
# Client.login mutates the client's bearer token in place.
op_home = Client(BASE)
op_home.login(email, password)

In [ ]:
org = op_home.tenants.create_organisation(
    CreateOrganisationRequest(name=f'AcmeCorp-{int(time.time())}', business='Technology')
)
print('created org', org.id)

## Switch into the tenant org and create a service account

In [ ]:
# switch_org returns a fresh JWT scoped to the target org. We open a separate
# Client for the tenant context so op_home keeps its operator-org permissions
# (used later for the feature override).
switch = op_home.security.switch_org(SwitchOrgRequest(org_id=org.id))
op_target = Client(BASE, jwt=switch.token)

sa = op_target.security.create_service_account(
    CreateServiceAccountRequest(name='echo-bot', organisation_id=org.id)
)

# Fresh service accounts have no membership row yet — grant one before
# the SA can authenticate against any org-scoped endpoint.
op_target.tenants.add_user_to_organisation(
    AddUserToOrganisationRequest(org_id=org.id, user_id=sa.user_id, role_code='org_admin')
)

key = op_target.security.create_api_key(
    sa.user_id,
    CreateApiKeyRequest(name='scenario-key', scopes=['echo:invoke']),
)
print('minted key', key.key.id)

## Positive case — payload round-trips through `/api/v1/echo`

In [ ]:
# Auth via API key (no JWT) — the typical service-account integration path.
caller = Client(BASE, api_key=key.plaintext)
echoed = caller.echo.echo_post({'hello': 'world'})

assert echoed.method == 'POST'
assert echoed.payload == {'hello': 'world'}
assert echoed.org_id == org.id
assert echoed.key_id == key.key.id
assert echoed.principal_user_id == sa.user_id
print('echo round-trip OK', echoed)

## Negative case — a key without `echo:invoke` is rejected with 403

In [ ]:
no_perm = op_target.security.create_api_key(
    sa.user_id,
    CreateApiKeyRequest(name='no-echo', scopes=['features.read']),
)

# The typed client raises Forbidden for HTTP 403; the parsed RFC 7807 problem
# body is on .problem and the slug/type round-trips.
try:
    Client(BASE, api_key=no_perm.plaintext).echo.echo_post({'x': 1})
    raise AssertionError('expected 403')
except Forbidden as e:
    assert (e.problem.type or '').endswith('/permission.denied'), e.problem
    print('rejected without echo:invoke OK —', e)

## Header allowlist demo — restrict the org to `X-API-Key` only

The flag `auth.api_key_headers` is not self-service, so the operator (with `tenants.manage_all`) sets the override from their home-org JWT (`op_home`). Once set, the same API key sent via `Authorization: Bearer …` is rejected with 401, while `X-API-Key: …` continues to work.

The OpenAPI spec types `PutFeatureRequest.value` as `object`, but the server actually accepts any JSON value (bools, strings, lists). To stay correct without lying to Pydantic, we pass a raw dict here (the api method accepts either a typed model or a dict).

In [ ]:
op_home.features.put_org_feature(
    org.id, 'auth.api_key_headers',
    {'value': ['x-api-key']},
)

# Round-trip via Authorization: Bearer — should be rejected.
via_bearer = httpx.post(
    f'{BASE}/api/v1/echo',
    headers={'authorization': f'Bearer {key.plaintext}', 'content-type': 'application/json'},
    json={'should': 'fail'},
)
assert via_bearer.status_code == 401, via_bearer.text

# Same key via X-API-Key — should still work.
via_x_api_key = caller.echo.echo_post({'still': 'works'})
print('header allowlist enforced — bearer 401, x-api-key 200', via_x_api_key)